In [62]:
from pathlib import Path
from datetime import datetime
from urllib.request import urlopen
import hashlib
import pandas as pd

In [63]:
url_fonte = 'https://claytonjasilva.github.io/progLinear/trabalhoap1/Problema1_Financas_DadosBCB.xlsx'

In [64]:
destino = Path("..") / "dados"
destino.mkdir(parents=True, exist_ok=True)

# Calcular o hash SHA-256 do arquivo usado
def calcular_hash(caminho):
    sha256 = hashlib.sha256()
    with caminho.open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(8192), b""):
            sha256.update(bloco)
    return sha256.hexdigest()

arquivos_locais = sorted(
    destino.glob("Problema1_Financas_DadosBCB_*.xlsx"),
    key=lambda caminho: caminho.stat().st_mtime,
    reverse=True,
)

if arquivos_locais:
    caminho_planilha = arquivos_locais[0]
    arquivo_hash = Path(f"{caminho_planilha}.sha256")
    hash_original = calcular_hash(caminho_planilha)
    hash_armazenado = arquivo_hash.read_text(encoding="ascii").strip() if arquivo_hash.exists() else ""

    if hash_original == hash_armazenado:
        origem_arquivo = "Arquivo local reutilizado"
    else:
        caminho_planilha = None

if not arquivos_locais or caminho_planilha is None:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    caminho_planilha = destino / f"Problema1_Financas_DadosBCB_{timestamp}.xlsx"

    # Baixar uma nova versão quando não houver cópia válida local
    with urlopen(url_fonte) as resposta:
        caminho_planilha.write_bytes(resposta.read())

    hash_original = calcular_hash(caminho_planilha)
    arquivo_hash = Path(f"{caminho_planilha}.sha256")
    arquivo_hash.write_text(f"{hash_original}\n", encoding="ascii")
    origem_arquivo = "Arquivo baixado"

# Conferir o hash novamente após reler o arquivo
hash_verificado = calcular_hash(caminho_planilha)

print(f"{origem_arquivo}: {caminho_planilha.resolve()}")
print(f"Tamanho: {caminho_planilha.stat().st_size} bytes")
print(f"Hash salvo em: {arquivo_hash.resolve()}")
print("Hash funcionando corretamente:", hash_original == hash_verificado)

Arquivo local reutilizado: /workspaces/PesquisaOperacional-otimizacao-de-investimento-em-ativos/dados/Problema1_Financas_DadosBCB_20261006_123832.xlsx
Tamanho: 44552 bytes
Hash salvo em: /workspaces/PesquisaOperacional-otimizacao-de-investimento-em-ativos/dados/Problema1_Financas_DadosBCB_20261006_123832.xlsx.sha256
Hash funcionando corretamente: True


In [65]:
df_geral = pd.read_excel(caminho_planilha, sheet_name=None)

In [66]:
def inspecao(dataframe):
    """Retorna a inspeção das colunas de um DataFrame."""
    total_linhas = len(dataframe)
    linhas = []

    for indice, nome_coluna in enumerate(dataframe.columns):
        coluna = dataframe.iloc[:, indice]
        linhas.append({
            "coluna": nome_coluna,
            "tipo": str(coluna.dtype),
            "não nulos": int(coluna.notna().sum()),
            "nulos": int(coluna.isna().sum()),
            "% nulos": round(coluna.isna().sum() / total_linhas * 100, 2)
            if total_linhas else 0,
            "amostra": coluna.dropna().head(5).tolist(),
        })

    return pd.DataFrame(linhas).set_index("coluna")

for nome, tabela in df_geral.items():
    relatorio_inspecao = inspecao(tabela)
    print(f"Relatório de inspeção para a aba: {nome}")
    display(relatorio_inspecao)

Relatório de inspeção para a aba: CDI_serie12


,tipo,não nulos,nulos,% nulos,amostra
coluna,,,,,
Data,str,921,0,0.0,"[02/01/2023, 03/01/2023, 04/01/2023, 05/01/202..."
Valor,float64,921,0,0.0,"[0.050788, 0.050788, 0.050788, 0.050788, 0.050..."


Relatório de inspeção para a aba: Selic_serie432


,tipo,não nulos,nulos,% nulos,amostra
coluna,,,,,
Data,str,1340,0,0.0,"[01/01/2023, 02/01/2023, 03/01/2023, 04/01/202..."
Valor,float64,1340,0,0.0,"[13.75, 13.75, 13.75, 13.75, 13.75]"


Relatório de inspeção para a aba: IPCA_serie433


,tipo,não nulos,nulos,% nulos,amostra
coluna,,,,,
Data,str,44,0,0.0,"[01/01/2023, 01/02/2023, 01/03/2023, 01/04/202..."
Valor,float64,44,0,0.0,"[0.53, 0.84, 0.71, 0.61, 0.23]"


Relatório de inspeção para a aba: Dolar_serie1


,tipo,não nulos,nulos,% nulos,amostra
coluna,,,,,
Data,str,921,0,0.0,"[02/01/2023, 03/01/2023, 04/01/2023, 05/01/202..."
Valor,float64,921,0,0.0,"[5.3436, 5.3759, 5.4459, 5.4026, 5.2855]"


In [67]:
display(inspecao(df_geral['CDI_serie12']))

,tipo,não nulos,nulos,% nulos,amostra
coluna,,,,,
Data,str,921,0,0.0,"[02/01/2023, 03/01/2023, 04/01/2023, 05/01/202..."
Valor,float64,921,0,0.0,"[0.050788, 0.050788, 0.050788, 0.050788, 0.050..."


In [68]:
for nome, dataframe in df_geral.items():
    linhas_duplicadas = dataframe["Data"].duplicated(keep="first")
    linhas_removidas = dataframe.loc[linhas_duplicadas].copy()

    df_geral[nome] = (
        dataframe.loc[~linhas_duplicadas]
        .reset_index(drop=True)
    )

    print(f"{nome}: {len(linhas_removidas)} linha(s) removida(s)")
    if not linhas_removidas.empty:
        display(linhas_removidas)

CDI_serie12: 0 linha(s) removida(s)
Selic_serie432: 0 linha(s) removida(s)
IPCA_serie433: 0 linha(s) removida(s)
Dolar_serie1: 0 linha(s) removida(s)


In [69]:
destino_modelo = Path("..") / "modelo"
destino_modelo.mkdir(parents=True, exist_ok=True)
arquivo_limpo = destino_modelo / "Financas_DadosBCB_clean.xlsx"

with pd.ExcelWriter(arquivo_limpo) as escritor:
    for nome, dataframe in df_geral.items():
        dataframe.to_excel(escritor, sheet_name=nome, index=False)

print(f"Planilha limpa salva em: {arquivo_limpo.resolve()}")

Planilha limpa salva em: /workspaces/PesquisaOperacional-otimizacao-de-investimento-em-ativos/modelo/Financas_DadosBCB_clean.xlsx


# Conclusão

A extração e inspeção demonstrou que os dados não possuem datas duplicadas, ou valores ausentes, o que conclui que base já está pronta para seguir para a próxima etapa.